# Clean and Preprocess Reddit Text
This notebook loads merged Reddit text, applies normalization + tokenization + stopword removal + lemmatization, creates `clean_text`, and saves a cleaned parquet file.

In [9]:
from pathlib import Path
import re

import pandas as pd

try:
    import nltk
except ImportError as exc:
    raise ImportError(
        "nltk is required for preprocessing. Install it in your active environment (for example: pip install nltk)."
    ) from exc

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()

In [10]:
# Resolve project paths and load merged dataset.
cwd = Path.cwd().resolve()
if (cwd / 'Data').exists() or (cwd / 'data').exists():
    project_root = cwd
elif (cwd.parent / 'Data').exists() or (cwd.parent / 'data').exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError('Could not find Data/ or data/ from current working directory.')

data_dir = project_root / 'Data' if (project_root / 'Data').exists() else project_root / 'data'
processed_dir = data_dir / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

input_path = processed_dir / 'reddit_mh_merged.parquet'
if not input_path.exists():
    raise FileNotFoundError(f'Input file not found: {input_path}')

df = pd.read_parquet(input_path)
text_col_candidates = ['post', 'text', 'body', 'content']
text_col = next((c for c in text_col_candidates if c in df.columns), None)
if text_col is None:
    raise KeyError(
        f"No text column found. Tried {text_col_candidates}. Available columns: {list(df.columns)[:20]}"
    )

print(f'Loaded shape: {df.shape}')
print(f'Using text column: {text_col}')

Loaded shape: (1107302, 353)
Using text column: post


In [11]:
# Cleaning and preprocessing pipeline.
url_re = re.compile(r'https?://\S+|www\.\S+')
reddit_ref_re = re.compile(r'\b(?:r|u)/[A-Za-z0-9_-]+\b')
special_re = re.compile(r'[^a-z0-9\s\']')
ws_re = re.compile(r'\s+')
token_re = re.compile(r"[a-z]+(?:'[a-z]+)?")

base_stopwords = set(stopwords.words('english'))
negations_to_keep = {'no', 'nor', 'not', 'never'}
stop_words = base_stopwords - negations_to_keep

def normalize_text(text: str) -> str:
    text = text.lower()
    text = url_re.sub(' ', text)
    text = reddit_ref_re.sub(' ', text)
    text = text.replace('[removed]', ' ').replace('[deleted]', ' ')
    text = text.replace('&amp;', ' and ').replace('&gt;', ' ').replace('&lt;', ' ')
    text = special_re.sub(' ', text)
    text = ws_re.sub(' ', text).strip()
    return text

def tokenize(text: str) -> list[str]:
    return token_re.findall(text)

def is_negation_token(token: str) -> bool:
    return token in negations_to_keep or token.endswith("n't")

def preprocess_text(text: str) -> tuple[str, list[str]]:
    normalized = normalize_text(text)
    tokens = tokenize(normalized)
    # Keep explicit negations and contraction negations even if present in stopwords.
    tokens = [t for t in tokens if (t not in stop_words) or is_negation_token(t)]
    lemmas = [lemmatizer.lemmatize(t) for t in tokens]
    clean_text = ' '.join(lemmas)
    return clean_text, lemmas

text_series = df[text_col].fillna('').astype(str)
processed = text_series.apply(preprocess_text)

df['clean_text'] = processed.str[0]
df['tokens'] = processed.str[1]
df['clean_word_count'] = df['tokens'].str.len()
df['source_text_col'] = text_col

print(df[['clean_text', 'clean_word_count', 'source_text_col']].head())

                                          clean_text  clean_word_count  \
0  welcome covid support hi everyone counselor mo...               110   
1  deadman walking type diabetes mid feel like vi...                46   
2  constructive action not pink glass disconcerti...               208   
3  one way dealt pandemic panic sars cov one way ...                92   
4  no one around care covid virus going live u so...                28   

  source_text_col  
0            post  
1            post  
2            post  
3            post  
4            post  


In [12]:
output_path = processed_dir / 'reddit_mh_clean.parquet'
df.to_parquet(output_path, index=False)

print(f'Saved cleaned dataset to: {output_path}')
print(f'Final shape: {df.shape}')

Saved cleaned dataset to: /Users/ganenthraravindran/Desktop/Mental Health/Data/processed/reddit_mh_clean.parquet
Final shape: (1107302, 357)
